# Machine Learning Assignment: Personal Loan Prediction

**Approach:** Supervised Learning  
**Algorithm:** Logistic Regression  
**Dataset:** Bank Personal Loan Modelling

This notebook follows the assignment rubric: dataset description, preprocessing, model justification, implementation, results, and discussion.

## 1. Dataset Description

- Source file: `dataset/Bank_Personal_Loan_Modelling.csv`
- Rows: 5,000
- Columns: 14
- Target variable: `Personal Loan` (0 = No, 1 = Yes)
- This is a binary classification problem: predict whether a customer accepts a personal loan offer.

In [ ]:
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)
RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('../dataset/Bank_Personal_Loan_Modelling.csv')
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

In [ ]:
print('Missing values by column:')
print(df.isnull().sum())

print('\nTarget distribution:')
print(df['Personal Loan'].value_counts())

## 2. Data Preprocessing

Preprocessing steps:
- Remove identifier-like columns (`ID`, `ZIP Code`) from features.
- Handle impossible negative values in `Experience` by clipping at 0.
- Standardize numeric variables.
- One-hot encode categorical/discrete variables.
- Use stratified train-test split due to class imbalance.

In [ ]:
df['Experience'] = df['Experience'].clip(lower=0)

X = df.drop(columns=['Personal Loan', 'ID', 'ZIP Code'])
y = df['Personal Loan']

numeric_features = ['Age', 'Experience', 'Income', 'CCAvg', 'Mortgage']
categorical_features = ['Family', 'Education', 'Securities Account', 'CD Account', 'Online', 'CreditCard']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))]), categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_train.shape, X_test.shape

## 3. Model Selection and Justification

**Logistic Regression** is appropriate because:
- The target is binary (`Personal Loan` = 0/1).
- It provides interpretable coefficients and probability outputs.
- It is an efficient baseline for tabular classification tasks.

In [ ]:
model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=3000,
    solver='liblinear'
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

pipeline.fit(X_train, y_train)

## 4. Results

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-score': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_prob),
}

metrics

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix [[TN, FP], [FN, TP]]:')
print(cm)

print('\nClassification Report:')
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_auc = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc')

print(f'5-fold CV ROC-AUC mean: {cv_auc.mean():.4f}')
print(f'5-fold CV ROC-AUC std : {cv_auc.std():.4f}')

## 5. Additional Discussion (Original Notes)

Key observations:
- Accuracy is high, but class imbalance means precision/recall are more informative than accuracy alone.
- False negatives (missed potential borrowers) exist and can be reduced by threshold tuning or class weighting.

Possible improvements:
- Compare with additional supervised algorithms (e.g., Decision Tree, Random Forest, SVM/XGBoost).
- Perform hyperparameter optimization with cross-validation.
- Explore feature engineering and probability threshold optimization based on business goals.

## 6. Training Experiments and Metric Changes

This section shows how performance changes with model/training decisions:
- regularization strength (`C`)
- class weighting
- decision threshold
- training set size

In [ ]:
def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-score': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_prob),
    }

settings = [
    {'setting': 'baseline', 'C': 1.0, 'class_weight': None, 'threshold': 0.50},
    {'setting': 'stronger_reg', 'C': 0.1, 'class_weight': None, 'threshold': 0.50},
    {'setting': 'weaker_reg', 'C': 5.0, 'class_weight': None, 'threshold': 0.50},
    {'setting': 'balanced_weight', 'C': 1.0, 'class_weight': 'balanced', 'threshold': 0.50},
    {'setting': 'threshold_0.35', 'C': 1.0, 'class_weight': None, 'threshold': 0.35},
]

rows = []
for s in settings:
    model = LogisticRegression(random_state=RANDOM_STATE, max_iter=3000, solver='liblinear', C=s['C'], class_weight=s['class_weight'])
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= s['threshold']).astype(int)
    m = evaluate_predictions(y_test, y_pred, y_prob)
    rows.append({
        'setting': s['setting'],
        'C': s['C'],
        'class_weight': s['class_weight'] if s['class_weight'] is not None else 'None',
        'threshold': s['threshold'],
        **m,
    })

settings_df = pd.DataFrame(rows).sort_values('F1-score', ascending=False)
settings_df

In [ ]:
train_ratios = [0.5, 0.6, 0.7, 0.8]
rows = []
for tr in train_ratios:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=(1-tr), stratify=y, random_state=RANDOM_STATE
    )
    model = LogisticRegression(random_state=RANDOM_STATE, max_iter=3000, solver='liblinear')
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_tr, y_tr)

    y_prob = pipeline.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    m = evaluate_predictions(y_te, y_pred, y_prob)
    rows.append({
        'train_ratio': tr,
        'train_samples': len(X_tr),
        'test_samples': len(X_te),
        **m,
    })

train_size_df = pd.DataFrame(rows)
train_size_df

## 7. Critical Analysis and Discussion

Key observations:
- Lower threshold can improve recall and F1 when missing positive cases is costly.
- Class weighting greatly increases recall but may reduce precision and overall accuracy.
- Performance is fairly stable across different train sizes, suggesting good generalization for this dataset.

## 8. Assignment Compliance Note

According to the assignment sheet, supervised-learning teams of 4 are expected to apply **4 distinct algorithms**.

This notebook covers Logistic Regression only. If your group must satisfy that rule, add 3 more algorithms and compare using the same metrics.